In [1]:
import subprocess
import sys
import os
import time

print("✓ Task 1: Kaggle environment ready")
print(f"  GPU available: {os.path.exists('/dev/nvidia-smi')}")

✓ Task 1: Kaggle environment ready
  GPU available: False


In [2]:
subprocess.run(
    ["git", "clone", "https://github.com/JamilProg/crosslingual_bert_annotation_projection.git"],
    cwd="/kaggle/working",
    capture_output=True
)

sys.path.insert(0, "/kaggle/working/crosslingual_bert_annotation_projection")

print("✓ Task 2: Repository cloned successfully")

✓ Task 2: Repository cloned successfully


In [3]:
print("Installing zstd dependency...")
subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
subprocess.run(["apt-get", "install", "-y", "zstd"], capture_output=True)


print("Installing Ollama... (this may take a minute)")
subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    capture_output=True
)

Installing zstd dependency...
Installing Ollama... (this may take a minute)


CompletedProcess(args='curl -fsSL https://ollama.com/install.sh | sh', returncode=0, stdout=b'\x1b\x1bWARNING:\x1b[m systemd is not running\n\x1b\x1bWARNING:\x1b[m Unable to detect NVIDIA/AMD GPU. Install lspci or lshw to automatically detect and install GPU dependencies.\n', stderr=b'>>> Installing ollama to /usr/local\n>>> Downloading ollama-linux-amd64.tar.zst\n#=#=#                                                                         \r##O#-#                                                                        \r\r                                                                           0.0%\r                                                                           0.1%\r                                                                           0.1%\r                                                                           0.2%\r                                                                           0.4%\r                                                                  

In [4]:
subprocess.run([sys.executable, "-m", "pip", "install", "ollama", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "tiktoken", "-q"])
print("✓ Task 3: Ollama and dependencies installed")

✓ Task 3: Ollama and dependencies installed


In [5]:
import subprocess
import time

# Start Ollama server in the background
print("Starting Ollama server...")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Give Ollama time to start
time.sleep(5)

Starting Ollama server...


In [6]:
try:
    import requests
    response = requests.get("http://localhost:11434/api/tags", timeout=5)
    print("✓ Task 4: Ollama server started successfully")
except Exception as e:
    print(f"⚠️ Ollama server startup issue: {e}")
    print("  Attempting to reconnect...")
    time.sleep(3)

✓ Task 4: Ollama server started successfully


In [7]:
print("Pulling Ministral 3B model from Ollama... (this may take 2-5 minutes)")

try:
    import ollama
    
    # Pull the model
    ollama.pull("ministral-3:3b")
    
    print("✓ Task 5: Model pulled successfully - ministral-3:3b")
except Exception as e:
    print(f"Error pulling model: {e}")
    print("Attempting alternative approach...")
    subprocess.run(["ollama", "pull", "ministral-3:3b"], capture_output=True)
    print("✓ Task 5: Model pulled via subprocess")


Pulling Ministral 3B model from Ollama... (this may take 2-5 minutes)
✓ Task 5: Model pulled successfully - ministral-3:3b


In [8]:
BASE_PATH = "/kaggle/working/crosslingual_bert_annotation_projection/data/input/DISTEMIST-FR"
print("✓ Task 6: Data paths configured")
print(f"  Base path: {BASE_PATH}")

✓ Task 6: Data paths configured
  Base path: /kaggle/working/crosslingual_bert_annotation_projection/data/input/DISTEMIST-FR


In [9]:
def load_medical_notes(folder_path):
    """
    Load all medical notes from a folder.
    
    Args:
        folder_path: Path to folder containing .txt files
    
    Returns:
        List of dictionaries with 'id' and 'text' keys
    """
    notes = []
    
    if not os.path.exists(folder_path):
        print(f"Warning: {folder_path} does not exist")
        return notes
    
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)
            
            try:
                with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read().strip()
                    
                    if len(text) > 20:
                        notes.append({
                            "id": filename.replace(".txt", ""),
                            "text": text
                        })
            except Exception as e:
                print(f"Error reading {filename}: {e}")
    
    return notes

# Load notes
notes = load_medical_notes(BASE_PATH)

print("✓ Task 7: Medical notes loaded")
print(f"  Total notes loaded: {len(notes)}")
if notes:
    print(f"\n  Sample note preview:\n  {notes[0]['text'][:100]}...")


✓ Task 7: Medical notes loaded
  Total notes loaded: 750

  Sample note preview:
  Patient âgé de 70 ans, mineur retraité, sans allergie médicamenteuse connue, avec pour antécédents p...


In [10]:
import tiktoken
import numpy as np

enc = tiktoken.get_encoding("gpt2")

print("✓ Task 8: Tokenizer initialized (GPT-2)")

✓ Task 8: Tokenizer initialized (GPT-2)


In [11]:
prompt_template_single = """
Analyse le texte ci-dessous et renvoie uniquement ce JSON strict :

{
  "text_id": "...",
  "annotations": {
    "sexe": null,
    "age": null,
    "profession": null,
    "niveau_intelectuel": null,
    "niveau_social": null,
    "situation_familiale": null,
    "symptomes": [],
    "ANTCD_Perso": [],
    "ANTCD_Famille": [],
    "resultat_examen": [],
    "resultat_scanner": [],
    "diagnostique": []
  }
}

Règles :
- "Patient" → "Homme", "Patiente" → "Femme"
- info absente → null ou []
- ne rien inventer
Texte :
"{text}"
"""

MAX_TOKENS = 2000

print("✓ Task 9: Prompt template defined")

✓ Task 9: Prompt template defined


In [12]:
def token_count(text):
    """Count tokens in a prompt with given text."""
    full_prompt = prompt_template_single.replace("{text}", text)
    return len(enc.encode(full_prompt))

print("✓ Task 10: Token counting function ready")

✓ Task 10: Token counting function ready


In [13]:
classe1 = []  # Large texts (>=2000 tokens)
classe2_candidates = []  # Small texts (<2000 tokens)

for n in notes:
    length = token_count(n['text'])
    n['tokens'] = length
    
    if length >= MAX_TOKENS:
        classe1.append([n])
    else:
        classe2_candidates.append(n)

print("✓ Task 11: Notes classified by size")
print(f"  Class 1 (large texts): {len(classe1)} batches")
print(f"  Small text candidates: {len(classe2_candidates)}")

✓ Task 11: Notes classified by size
  Class 1 (large texts): 33 batches
  Small text candidates: 717


In [14]:
classe2 = []

i = 0
while i < len(classe2_candidates):
    batch = [classe2_candidates[i]]
    batch_tokens = classe2_candidates[i]['tokens']
    
    j = i + 1
    while j < len(classe2_candidates) and len(batch) < 3:
        if batch_tokens + classe2_candidates[j]['tokens'] <= MAX_TOKENS:
            batch.append(classe2_candidates[j])
            batch_tokens += classe2_candidates[j]['tokens']
        j += 1
    
    if len(batch) == 1 and batch_tokens >= MAX_TOKENS:
        classe1.append(batch)
    else:
        classe2.append(batch)
    
    i += len(batch)

print("✓ Task 12: Batching completed")
print(f"  Class 2 batches: {len(classe2)}")
print(f"  Total batches in Class 1: {len(classe1)}")

✓ Task 12: Batching completed
  Class 2 batches: 360
  Total batches in Class 1: 33


In [15]:
def create_batch_prompt(batch):
    """Create a prompt for multiple texts in one batch."""
    prompt_start = """
        Tu es un annotateur médical automatique.  
        Annoter plusieurs textes médicaux **séparément**.  
        Chaque texte doit avoir son propre objet JSON strict selon le schéma :
        {
          "text_id": "...",
          "annotations": {
            "sexe": null,
            "age": null,
            "profession": null,
            "niveau_intelectuel": null,
            "niveau_social": null,
            "situation_familiale": null,
            "symptomes": [],
            "ANTCD_Perso": [],
            "ANTCD_Famille": [],
            "resultat_examen": [],
            "resultat_scanner": [],
            "diagnostique": []
          }
        }
        
        Règles :
        - "Patient" → "Homme", "Patiente" → "Femme"
        - info absente → null ou []
        - ne rien inventer
        - **Ne jamais mélanger les informations entre les textes**
        - Répond uniquement en JSON avec un objet pour chaque texte
    """
    texts_part = ""
    for idx, n in enumerate(batch, 1):
        texts_part += f"\nTexte {idx} (ID: {n['id']}):\n\"\"\"\n{n['text']}\n\"\"\"\n"
    
    return prompt_start + texts_part

print("✓ Task 13: Batch prompt function ready")



✓ Task 13: Batch prompt function ready


In [16]:
prompts_classe1 = []
for batch in classe1:
    prompt = create_batch_prompt(batch)
    prompts_classe1.append(prompt)

prompts_classe2 = []
for batch in classe2:
    prompt = create_batch_prompt(batch)
    prompts_classe2.append(prompt)

print("✓ Task 14: All prompts generated")
print(f"\n  Summary:")
print(f"  ├─ Class 1 batches: {len(prompts_classe1)}")
print(f"  ├─ Class 2 batches: {len(prompts_classe2)}")
print(f"  └─ Total batches: {len(prompts_classe1) + len(prompts_classe2)}")

✓ Task 14: All prompts generated

  Summary:
  ├─ Class 1 batches: 33
  ├─ Class 2 batches: 360
  └─ Total batches: 393


In [17]:
import ollama

try:
    # Test with a simple query
    test_response = ollama.chat(
        model='ministral-3:3b',
        messages=[
            {'role': 'user', 'content': 'Say "OK" in one word only'}
        ],
        stream=False
    )
    print("✓ Task 15: Ollama connection successful")
    print(f"  Test response: {test_response['message']['content']}")
except Exception as e:
    print(f"⚠️ Task 15: Ollama connection issue: {e}")

✓ Task 15: Ollama connection successful
  Test response: OK


In [18]:
def annotate_batch_with_ollama(prompt, model='ministral-3:3b'):
    """
    Send a batch prompt to Ollama and get annotations.
    
    Args:
        prompt: The formatted batch prompt
        model: Ollama model name
    
    Returns:
        Response text from the model
    """
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {'role': 'user', 'content': prompt}
            ],
            stream=False
        )
        return response['message']['content']
    except Exception as e:
        print(f"Error in annotation: {e}")
        return None

In [19]:
if prompts_classe1:
    print("✓ Task 16: Testing annotation on first batch...")
    print(f"  Prompt size: {len(enc.encode(prompts_classe1[0]))} tokens")
    
    result = annotate_batch_with_ollama(prompts_classe1[0])
    
    if result:
        print(f"  Result preview:\n  {result[:200]}...")
    else:
        print("  Annotation failed - check Ollama connection")

✓ Task 16: Testing annotation on first batch...
  Prompt size: 2753 tokens
  Result preview:
  ```json
{
  "text_id": "es-S0210-48062003001000009-1",
  "annotations": {
    "sexe": "Femme",
    "age": 35,
    "profession": null,
    "niveau_intelectuel": null,
    "niveau_social": null,
    "si...


In [20]:
# Prendre le premier batch de classe2
example_prompt2 = prompts_classe2[0]

# Sauvegarder le prompt dans un fichier temporaire
with open("prompt_classe2.txt", "w", encoding="utf-8") as f:
    f.write(example_prompt2)

# Appeler Ollama en CLI pour deepseek
!ollama run ministral-3:3b "$(cat prompt_classe2.txt)"

⠙ ⠙ ⠸ ⠸ ⠴ ⠴ ⠦ ⠧ ```json
[
  {
    "text_id": "S0004-06142005000500011-1",
    "annotations": {
      "sexe": "Homme",
      "age": 70,
      "profession": null,
      "niveau_intelectuel": null,
      "niveau_social": null,
      "situation_familiale": null,
      "symptomes": ["hématurie macroscopique postmictionnelle", "microhémat
"microhématurie persistante"],
      "ANTCD_Perso": [
        "ancien accident du travail avec fractures des vertèbres et des côt
côtes",
        "opéré pour une maladie de Dupuytren à la main droite",
        "by-pass ilio-fémoral gauche",
        "diabète sucré de type II",
        "hypercholestérolémie",
        "hyperuricémie",
        "alcoolisme actif",
        "fumeur de 20 cigarettes/jour"
      ],
      "ANTCD_Famille": [],
      "resultat_examen": [
        "bon état général",
        "abdomen et organes génitaux normaux",
        "toucher rectal compatible avec un adénome de la prostate de grade 
I/IV"
      ],
      "resultat_scanner": [
       

In [21]:
import json

config = {
    "total_notes": len(notes),
    "class1_batches": len(prompts_classe1),
    "class2_batches": len(prompts_classe2),
    "model": "ministral-3:3b",
    "max_tokens": MAX_TOKENS
}

with open("/kaggle/working/annotation_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✓ Task 17: Configuration saved")
print(f"\n✅ Pipeline setup complete!")
print(f"   Ready to process {config['total_notes']} notes")
print(f"   Using {config['class1_batches'] + config['class2_batches']} batches")

✓ Task 17: Configuration saved

✅ Pipeline setup complete!
   Ready to process 750 notes
   Using 393 batches


In [22]:
import json
import re
import os
from pathlib import Path

In [23]:
output_dir = "/kaggle/working/annotations"
Path(output_dir).mkdir(parents=True, exist_ok=True)

print(f"✓ Created output directory: {output_dir}")

✓ Created output directory: /kaggle/working/annotations


In [24]:
def fix_json_string(json_str):
    """
    Attempt to fix common JSON formatting errors.
    
    Args:
        json_str: Potentially malformed JSON string
    
    Returns:
        Tuple: (fixed_json_string, is_valid_bool)
    """
    
    # Try to parse as-is first
    try:
        json.loads(json_str)
        return json_str, True
    except json.JSONDecodeError:
        pass
    
    # Remove markdown code blocks if present
    json_str = re.sub(r'```json\s*', '', json_str)
    json_str = re.sub(r'```\s*$', '', json_str)
    
    # Try again after markdown removal
    try:
        json.loads(json_str)
        return json_str, True
    except json.JSONDecodeError:
        pass
    
    # Fix common issues
    # 1. Replace smart quotes with regular quotes
    json_str = json_str.replace('"', '"').replace('"', '"')
    json_str = json_str.replace(''', "'").replace(''', "'")
    
    # 2. Fix missing commas between key-value pairs
    json_str = re.sub(r'"\s*\n\s*"', '",\n"', json_str)
    
    # 3. Remove trailing commas
    json_str = re.sub(r',(\s*[}\]])', r'\1', json_str)
    
    # 4. Fix newlines inside strings (escape them)
    # This is tricky, so we try a conservative approach
    
    try:
        json.loads(json_str)
        return json_str, True
    except json.JSONDecodeError:
        pass
    
    # 5. Try to extract JSON object if there's surrounding text
    match = re.search(r'\{.*\}', json_str, re.DOTALL)
    if match:
        extracted = match.group(0)
        try:
            json.loads(extracted)
            return extracted, True
        except json.JSONDecodeError:
            pass
    
    # 6. If still failing, try fixing unquoted keys (very basic)
    try:
        # Replace unquoted property names
        fixed = re.sub(r'([{,]\s*)([a-zA-Z_][a-zA-Z0-9_]*)\s*:', r'\1"\2":', json_str)
        json.loads(fixed)
        return fixed, True
    except json.JSONDecodeError:
        pass
    
    return json_str, False


def validate_and_parse_json(response_text):
    """
    Parse JSON from model response with auto-correction.
    
    Args:
        response_text: Raw text from Ollama model
    
    Returns:
        List of valid JSON objects or empty list if parsing failed
    """
    
    fixed_text, is_valid = fix_json_string(response_text)
    
    if is_valid:
        try:
            # Try to parse as single object
            parsed = json.loads(fixed_text)
            if isinstance(parsed, dict):
                return [parsed]
            elif isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            pass
    
    # Try to extract multiple JSON objects
    json_objects = []
    depth = 0
    current_obj = ""
    in_string = False
    escape_next = False
    
    for char in fixed_text:
        if escape_next:
            current_obj += char
            escape_next = False
            continue
        
        if char == '\\':
            current_obj += char
            escape_next = True
            continue
        
        if char == '"' and not escape_next:
            in_string = not in_string
        
        if not in_string:
            if char == '{':
                depth += 1
            elif char == '}':
                depth -= 1
        
        current_obj += char
        
        if depth == 0 and current_obj.strip():
            try:
                obj = json.loads(current_obj)
                json_objects.append(obj)
                current_obj = ""
            except json.JSONDecodeError:
                pass
    
    return json_objects

print("✓ Task 2: JSON validation and correction function ready")


✓ Task 2: JSON validation and correction function ready


In [25]:

import ollama
import time

results = []  # Main results list
failed_batches = []

print("\n" + "="*60)
print("PROCESSING ALL BATCHES WITH OLLAMA")
print("="*60)

# Combine both classes for processing
all_batches = [(batch, prompts_classe1[i], 'classe1') 
               for i, batch in enumerate(classe1)] + \
              [(batch, prompts_classe2[i], 'classe2') 
               for i, batch in enumerate(classe2)]

total_batches = len(all_batches)
print(f"\nTotal batches to process: {total_batches}")
print(f"├─ Class 1: {len(classe1)}")
print(f"└─ Class 2: {len(classe2)}\n")

for batch_idx, (batch, prompt, batch_class) in enumerate(all_batches, 1):
    
    # Progress indicator
    progress = f"[{batch_idx}/{total_batches}]"
    texts_in_batch = len(batch)
    print(f"{progress} Processing {batch_class} batch with {texts_in_batch} text(s)...", end=" ")
    
    try:
        # Call Ollama
        response = ollama.chat(
            model='ministral-3:3b',
            messages=[
                {'role': 'user', 'content': prompt}
            ],
            stream=False
        )
        
        response_text = response['message']['content']
        
        # Validate and parse JSON
        parsed_objects = validate_and_parse_json(response_text)
        
        if parsed_objects:
            results.extend(parsed_objects)
            print(f"✓ Success ({len(parsed_objects)} object(s) extracted)")
        else:
            print(f"⚠️ Warning: Could not parse JSON from response")
            failed_batches.append({
                'batch_idx': batch_idx,
                'batch_class': batch_class,
                'texts': [t['id'] for t in batch],
                'raw_response': response_text[:200]
            })
    
    except Exception as e:
        print(f"✗ Error: {str(e)[:50]}")
        failed_batches.append({
            'batch_idx': batch_idx,
            'batch_class': batch_class,
            'texts': [t['id'] for t in batch],
            'error': str(e)
        })
    
    # Small delay to prevent overwhelming Ollama
    time.sleep(0.5)

print("\n" + "="*60)
print(f"PROCESSING COMPLETE")
print("="*60)
print(f"✓ Total results extracted: {len(results)}")
print(f"⚠️ Failed batches: {len(failed_batches)}")




PROCESSING ALL BATCHES WITH OLLAMA

Total batches to process: 393
├─ Class 1: 33
└─ Class 2: 360

[1/393] Processing classe1 batch with 1 text(s)... ✓ Success (1 object(s) extracted)
[2/393] Processing classe1 batch with 1 text(s)... ✓ Success (1 object(s) extracted)
[3/393] Processing classe1 batch with 1 text(s)... ✓ Success (1 object(s) extracted)
[4/393] Processing classe1 batch with 1 text(s)... ✓ Success (1 object(s) extracted)
[5/393] Processing classe1 batch with 1 text(s)... ✓ Success (1 object(s) extracted)
[6/393] Processing classe1 batch with 1 text(s)... ✓ Success (1 object(s) extracted)
[7/393] Processing classe1 batch with 1 text(s)... ✓ Success (1 object(s) extracted)
[8/393] Processing classe1 batch with 1 text(s)... ✓ Success (1 object(s) extracted)
[9/393] Processing classe1 batch with 1 text(s)... ✓ Success (1 object(s) extracted)
[10/393] Processing classe1 batch with 1 text(s)... ✓ Success (1 object(s) extracted)
[11/393] Processing classe1 batch with 1 text(s)..

In [26]:
def validate_annotation_structure(obj):
    """
    Validate that annotation object has correct structure.
    
    Args:
        obj: Dictionary to validate
    
    Returns:
        Boolean: True if valid structure
    """
    required_fields = ['text_id', 'annotations']
    
    if not isinstance(obj, dict):
        return False
    
    if not all(field in obj for field in required_fields):
        return False
    
    annotations = obj.get('annotations', {})
    if not isinstance(annotations, dict):
        return False
    
    return True

valid_results = []
invalid_results = []

for idx, result in enumerate(results):
    if validate_annotation_structure(result):
        valid_results.append(result)
    else:
        invalid_results.append({
            'index': idx,
            'object': result
        })

print(f"\n✓ Valid annotations: {len(valid_results)}")
print(f"⚠️ Invalid annotations: {len(invalid_results)}")

if invalid_results:
    print("\nInvalid annotations details:")
    for invalid in invalid_results[:3]:  # Show first 3
        print(f"  Index {invalid['index']}: {str(invalid['object'])[:100]}")



✓ Valid annotations: 618
⚠️ Invalid annotations: 0


In [27]:
saved_count = 0
save_errors = []

print(f"\nSaving {len(valid_results)} annotations to individual JSON files...")

for result in valid_results:
    try:
        text_id = result.get('text_id', 'unknown')
        
        # Sanitize filename
        safe_filename = re.sub(r'[<>:"/\\|?*]', '_', str(text_id))
        file_path = os.path.join(output_dir, f"{safe_filename}.json")
        
        # Write JSON file with pretty formatting
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        
        saved_count += 1
        
        if saved_count % 10 == 0:
            print(f"  └─ Saved {saved_count} files...")
    
    except Exception as e:
        save_errors.append({
            'text_id': result.get('text_id'),
            'error': str(e)
        })

print(f"✓ Saved {saved_count} JSON files to {output_dir}")
if save_errors:
    print(f"⚠️ Save errors: {len(save_errors)}")


Saving 618 annotations to individual JSON files...
  └─ Saved 10 files...
  └─ Saved 20 files...
  └─ Saved 30 files...
  └─ Saved 40 files...
  └─ Saved 50 files...
  └─ Saved 60 files...
  └─ Saved 70 files...
  └─ Saved 80 files...
  └─ Saved 90 files...
  └─ Saved 100 files...
  └─ Saved 110 files...
  └─ Saved 120 files...
  └─ Saved 130 files...
  └─ Saved 140 files...
  └─ Saved 150 files...
  └─ Saved 160 files...
  └─ Saved 170 files...
  └─ Saved 180 files...
  └─ Saved 190 files...
  └─ Saved 200 files...
  └─ Saved 210 files...
  └─ Saved 220 files...
  └─ Saved 230 files...
  └─ Saved 240 files...
  └─ Saved 250 files...
  └─ Saved 260 files...
  └─ Saved 270 files...
  └─ Saved 280 files...
  └─ Saved 290 files...
  └─ Saved 300 files...
  └─ Saved 310 files...
  └─ Saved 320 files...
  └─ Saved 330 files...
  └─ Saved 340 files...
  └─ Saved 350 files...
  └─ Saved 360 files...
  └─ Saved 370 files...
  └─ Saved 380 files...
  └─ Saved 390 files...
  └─ Saved 400 files.

In [28]:
summary = {
    "processing_summary": {
        "total_batches_processed": total_batches,
        "total_texts_processed": len(notes),
        "successful_batches": total_batches - len(failed_batches),
        "failed_batches": len(failed_batches),
    },
    "json_validation": {
        "total_extracted": len(results),
        "valid_annotations": len(valid_results),
        "invalid_annotations": len(invalid_results),
    },
    "file_saving": {
        "files_saved": saved_count,
      "save_errors": len(save_errors),
        "output_directory": output_dir
    },
    "failed_batches": failed_batches[:5]  # First 5 failures
}

summary_path = os.path.join(output_dir, "_summary.json")
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"\n✓ Summary saved to {summary_path}")



✓ Summary saved to /kaggle/working/annotations/_summary.json


In [29]:
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"\n📊 Results stored in: results (list with {len(results)} items)")
print(f"\n📁 Output directory: {output_dir}")
print(f"   ├─ {saved_count} annotation files (text_id.json)")
print(f"   └─ _summary.json (processing report)")

if valid_results:
    print(f"\n📋 Sample result structure:")
    sample = valid_results[0]
    print(f"   text_id: {sample.get('text_id')}")
    print(f"   annotations keys: {list(sample.get('annotations', {}).keys())}")

print(f"\n✅ Processing Complete!")


FINAL SUMMARY

📊 Results stored in: results (list with 618 items)

📁 Output directory: /kaggle/working/annotations
   ├─ 618 annotation files (text_id.json)
   └─ _summary.json (processing report)

📋 Sample result structure:
   text_id: es-S0210-48062003001000009-1
   annotations keys: ['sexe', 'age', 'profession', 'niveau_intelectuel', 'niveau_social', 'situation_familiale', 'symptomes', 'ANTCD_Perso', 'ANTCD_Famille', 'resultat_examen', 'resultat_scanner', 'diagnostique']

✅ Processing Complete!
